In [1]:
# نصب کتابخانه‌ها در صورت نیاز

import sys
import subprocess
import importlib.util

required_packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "pyarrow": "pyarrow",
    "sklearn": "scikit-learn",
    "joblib": "joblib",
}

for module_name, package_name in required_packages.items():
    if importlib.util.find_spec(module_name) is None:
        print(f"Installing {package_name} ...")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package_name]
        )

print("All required packages are available.")

All required packages are available.


In [2]:
from pathlib import Path
from itertools import combinations
import gc
import hashlib
import json
import os
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 180)

RANDOM_STATE_MODEL = 142
np.random.seed(42)

In [3]:
# Portable configuration (repository root or notebooks/ working directory)
REPO_ROOT = Path(os.environ.get("FORCE_REPO_ROOT", Path.cwd())).expanduser().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "notebooks").is_dir():
    raise FileNotFoundError(
        "Repository root not found. Open Jupyter from the repository root "
        "or notebooks/, or set FORCE_REPO_ROOT to the repo path."
    )
DATA_DIR = Path(os.environ.get("FORCE_DATA_DIR", str(REPO_ROOT / "data" / "raw"))).expanduser().resolve()
PREPARED_DIR = Path(os.environ.get("FORCE_PREPARED_DIR", str(REPO_ROOT / "data" / "prepared"))).expanduser().resolve()
RESULTS_ROOT = Path(os.environ.get("FORCE_RESULTS_DIR", str(REPO_ROOT / "outputs"))).expanduser().resolve()

TRAIN_PATH = DATA_DIR / "train.csv"
PROJECT_ROOT = RESULTS_ROOT
STAGE_01_DIR = PREPARED_DIR
FOLD_ASSIGNMENT_PATH = STAGE_01_DIR / "10_grouped_5fold_well_assignment.csv"
CLEAN_DATA_PATH = STAGE_01_DIR / "FORCE_train_cleaned_with_splits.parquet"
LEGACY_REFERENCE_GATE_PATH = (
    PROJECT_ROOT / "02A_legacy_RF_exact_reproduction" / "09_reproduction_gate.json"
)
OUTPUT_DIR = PROJECT_ROOT / "02B_legacy_RF_grouped_well_CV" / RUN_MODE
METRIC_DIR = OUTPUT_DIR / "fold_metrics"
PREDICTION_DIR = OUTPUT_DIR / "oof_predictions"
IMPORTANCE_DIR = OUTPUT_DIR / "feature_importance"
RESAMPLING_DIR = OUTPUT_DIR / "resampling_audits"
ARTIFACT_DIR = OUTPUT_DIR / "artifacts"
PLOT_DIR = OUTPUT_DIR / "plots"

for directory in [
    OUTPUT_DIR,
    METRIC_DIR,
    PREDICTION_DIR,
    IMPORTANCE_DIR,
    RESAMPLING_DIR,
    ARTIFACT_DIR,
    PLOT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

# مدل مقاله قبلی؛ بدون تغییر
RF_N_ESTIMATORS = 800
RF_MAX_DEPTH = 200
RF_MIN_SAMPLES_SPLIT = 5
RF_MIN_SAMPLES_LEAF = 2
RF_N_JOBS = -1

# مدل‌های 800-tree ممکن است بسیار بزرگ باشند.
SAVE_MODELS = True
SAVE_PREPROCESSORS = True
SAVE_OOF_PROBABILITIES = True
SKIP_COMPLETED_FOLDS = True

if not TRAIN_PATH.exists():
    raise FileNotFoundError(TRAIN_PATH)

if not (
    FOLD_ASSIGNMENT_PATH.exists()
    or CLEAN_DATA_PATH.exists()
):
    raise FileNotFoundError(
        "Neither grouped fold assignment nor cleaned parquet exists."
    )

print("RUN_MODE:", RUN_MODE)
print("Outer folds:", ACTIVE_OUTER_FOLDS)
print("Train:", TRAIN_PATH)
print("Output:", OUTPUT_DIR)
print("SAVE_MODELS:", SAVE_MODELS)

RUN_MODE: full
Outer folds: [1, 2, 3, 4, 5]
Train: <historical-desktop>/train.csv
Output: <historical-output-root>/02B_legacy_RF_grouped_well_CV/full
SAVE_MODELS: True


In [4]:
def detect_csv_format(
    file_path,
    num_lines=5,
):
    encodings = [
        "utf-8",
        "latin1",
        "iso-8859-1",
        "cp1252",
    ]

    for encoding in encodings:
        try:
            with open(
                file_path,
                "r",
                encoding=encoding,
            ) as file:
                first_lines = [
                    file.readline()
                    for _ in range(num_lines)
                ]

            if any(
                "," in line
                for line in first_lines
            ):
                delimiter = ","
            elif any(
                ";" in line
                for line in first_lines
            ):
                delimiter = ";"
            elif any(
                "\t" in line
                for line in first_lines
            ):
                delimiter = "\t"
            else:
                delimiter = ","

            return {
                "sep": delimiter,
                "encoding": encoding,
            }

        except UnicodeDecodeError:
            continue

    return {
        "sep": ",",
        "encoding": "utf-8",
    }


def load_csv_full(
    file_path,
):
    csv_format = detect_csv_format(
        file_path
    )

    print(
        f"Loading {file_path.name} "
        f"with {csv_format}"
    )

    return pd.read_csv(
        file_path,
        sep=csv_format["sep"],
        encoding=csv_format["encoding"],
        low_memory=False,
    )


train_raw = load_csv_full(
    TRAIN_PATH
)

if "WELL" not in train_raw.columns:
    raise ValueError(
        "WELL column is missing from train.csv"
    )

if FOLD_ASSIGNMENT_PATH.exists():
    fold_assignment_raw = pd.read_csv(
        FOLD_ASSIGNMENT_PATH
    )

    well_candidates = [
        column
        for column in fold_assignment_raw.columns
        if column.upper() == "WELL"
    ]

    fold_candidates = [
        column
        for column in fold_assignment_raw.columns
        if (
            "FOLD" in column.upper()
            and "SCORE" not in column.upper()
        )
    ]

    if not well_candidates or not fold_candidates:
        raise ValueError(
            "Could not infer WELL/FOLD columns "
            "from fold assignment file."
        )

    assignment_well_column = (
        well_candidates[0]
    )

    assignment_fold_column = (
        fold_candidates[0]
    )

    fold_assignment = (
        fold_assignment_raw[
            [
                assignment_well_column,
                assignment_fold_column,
            ]
        ]
        .rename(
            columns={
                assignment_well_column: "WELL",
                assignment_fold_column: "GROUP_FOLD",
            }
        )
        .drop_duplicates(
            subset=["WELL"]
        )
        .copy()
    )

else:
    fold_assignment = (
        pd.read_parquet(
            CLEAN_DATA_PATH,
            columns=[
                "WELL",
                "GROUP_FOLD",
            ],
        )
        .drop_duplicates(
            subset=["WELL"]
        )
        .copy()
    )

fold_assignment[
    "GROUP_FOLD"
] = pd.to_numeric(
    fold_assignment[
        "GROUP_FOLD"
    ],
    errors="raise",
).astype(int)

well_to_fold = dict(
    zip(
        fold_assignment["WELL"],
        fold_assignment["GROUP_FOLD"],
    )
)

train_raw[
    "GROUP_FOLD"
] = train_raw[
    "WELL"
].map(
    well_to_fold
)

missing_fold_wells = sorted(
    train_raw.loc[
        train_raw["GROUP_FOLD"].isna(),
        "WELL",
    ]
    .dropna()
    .unique()
    .tolist()
)

if missing_fold_wells:
    raise RuntimeError(
        "Some train wells have no grouped fold assignment: "
        f"{missing_fold_wells[:20]}"
    )

train_raw[
    "GROUP_FOLD"
] = train_raw[
    "GROUP_FOLD"
].astype(int)

print("Train shape:", train_raw.shape)
print("Wells:", train_raw["WELL"].nunique())
print("Folds:", sorted(train_raw["GROUP_FOLD"].unique()))

fold_summary = (
    train_raw.groupby(
        "GROUP_FOLD"
    )
    .agg(
        row_count=(
            "WELL",
            "size",
        ),
        well_count=(
            "WELL",
            "nunique",
        ),
    )
    .reset_index()
)

display(fold_summary)

Loading train.csv with {'sep': ',', 'encoding': 'utf-8'}
Train shape: (1170511, 30)
Wells: 98
Folds: [1, 2, 3, 4, 5]


,GROUP_FOLD,row_count,well_count
0,1,232674,20
1,2,236205,20
2,3,230578,21
3,4,237132,17
4,5,233922,20


In [5]:
TARGET_COLUMN = (
    "FORCE_2020_LITHOFACIES_LITHOLOGY"
)

CONFIDENCE_COLUMN = (
    "FORCE_2020_LITHOFACIES_CONFIDENCE"
)

LITHOLOGY_CODES = sorted([
    30000,
    65030,
    65000,
    80000,
    74000,
    70000,
    70032,
    88000,
    86000,
    99000,
    90000,
    93000,
])

LITHOLOGY_NAMES = {
    30000: "Sandstone",
    65000: "Shale",
    65030: "Sandstone/Shale",
    70000: "Limestone",
    70032: "Chalk",
    74000: "Dolomite",
    80000: "Marl",
    86000: "Anhydrite",
    88000: "Halite",
    90000: "Coal",
    93000: "Basement",
    99000: "Tuff",
}

label_encoder = LabelEncoder()
label_encoder.fit(
    LITHOLOGY_CODES
)

CLASS_CODES = [
    int(value)
    for value in label_encoder.classes_
]

CLASS_INDICES = list(
    range(len(CLASS_CODES))
)

CLASS_INDEX_TO_CODE = {
    index: code
    for index, code
    in enumerate(CLASS_CODES)
}

CLASS_CODE_TO_INDEX = {
    code: index
    for index, code
    in CLASS_INDEX_TO_CODE.items()
}

CLASS_INDEX_TO_NAME = {
    index: LITHOLOGY_NAMES[code]
    for index, code
    in CLASS_INDEX_TO_CODE.items()
}

CHALK_INDEX = CLASS_CODE_TO_INDEX[70032]
SHALE_INDEX = CLASS_CODE_TO_INDEX[65000]

class_mapping = pd.DataFrame({
    "class_index": CLASS_INDICES,
    "lithology_code": CLASS_CODES,
    "lithology_name": [
        CLASS_INDEX_TO_NAME[index]
        for index in CLASS_INDICES
    ],
})

display(class_mapping)

,class_index,lithology_code,lithology_name
0,0,30000,Sandstone
1,1,65000,Shale
2,2,65030,Sandstone/Shale
3,3,70000,Limestone
4,4,70032,Chalk
5,5,74000,Dolomite
6,6,80000,Marl
7,7,86000,Anhydrite
8,8,88000,Halite
9,9,90000,Coal


In [ ]:
TOP_LEGACY_FEATURES = [
    "GR_NPHI_interaction",
    "GR_DTC_interaction",
    "Y_LOC",
    "GR_RHOB_RATIO",
    "NPHI_DTC_interaction",
    "X_LOC",
    "RHOB_DTC_interaction",
    "CALI",
    "Z_LOC",
    "GR_RHOB_interaction",
    "DEPTH_NORMALIZED",
    "DEPTH_MD",
    "RDEP",
    "RHOB_log",
    "RMED",
    "RHOB",
    "X_LOC",  
]

EXTRA_WEAK_CLASS_FEATURES = [
    "GR_flatness",
    "DTC_grad",
    "DTC_RHOB_DIFF",
    "DTC_RHOB_RATIO2",
    "RDEP_GR_RATIO",
    "RHOB_NPHI_DIFF2",
    "Acoustic_Impedance",
    "Sand_indicator",
    "Shale_indicator",
]

LEGACY_FEATURE_COLUMNS = (
    TOP_LEGACY_FEATURES
    + EXTRA_WEAK_CLASS_FEATURES
)

assert len(
    LEGACY_FEATURE_COLUMNS
) == 26

assert len(
    set(LEGACY_FEATURE_COLUMNS)
) == 25

assert (
    LEGACY_FEATURE_COLUMNS.count(
        "X_LOC"
    )
    == 2
)

print("Feature columns:", len(LEGACY_FEATURE_COLUMNS))
print("Unique names:", len(set(LEGACY_FEATURE_COLUMNS)))
print(LEGACY_FEATURE_COLUMNS)

Feature columns: 26
Unique names: 25
['GR_NPHI_interaction', 'GR_DTC_interaction', 'Y_LOC', 'GR_RHOB_RATIO', 'NPHI_DTC_interaction', 'X_LOC', 'RHOB_DTC_interaction', 'CALI', 'Z_LOC', 'GR_RHOB_interaction', 'DEPTH_NORMALIZED', 'DEPTH_MD', 'RDEP', 'RHOB_log', 'RMED', 'RHOB', 'X_LOC', 'GR_flatness', 'DTC_grad', 'DTC_RHOB_DIFF', 'DTC_RHOB_RATIO2', 'RDEP_GR_RATIO', 'RHOB_NPHI_DIFF2', 'Acoustic_Impedance', 'Sand_indicator', 'Shale_indicator']


In [7]:
def fit_and_apply_train_imputers(
    dataframe,
):
    df = dataframe.copy()

    if CONFIDENCE_COLUMN in df.columns:
        df = df.drop(
            columns=[CONFIDENCE_COLUMN]
        )

    numeric_columns = (
        df.select_dtypes(
            include=[np.number]
        )
        .columns
        .tolist()
    )

    for excluded in [
        TARGET_COLUMN,
        "LITHOLOGY_ENCODED",
        "GROUP_FOLD",
    ]:
        if excluded in numeric_columns:
            numeric_columns.remove(
                excluded
            )

    imputers = {}

    for column in numeric_columns:
        if df[column].isna().any():
            column_imputer = (
                SimpleImputer(
                    strategy="median"
                )
            )

            df[column] = (
                column_imputer
                .fit_transform(
                    df[[column]]
                )
                .ravel()
            )

            imputers[column] = (
                column_imputer
            )

    df[numeric_columns] = (
        df[numeric_columns]
        .fillna(0)
        .replace(
            [np.inf, -np.inf],
            0,
        )
    )

    return (
        df,
        imputers,
    )


def apply_train_imputers_to_validation(
    dataframe,
    imputers,
):
    df = dataframe.copy()

    if CONFIDENCE_COLUMN in df.columns:
        df = df.drop(
            columns=[CONFIDENCE_COLUMN]
        )

    numeric_columns = (
        df.select_dtypes(
            include=[np.number]
        )
        .columns
        .tolist()
    )

    for excluded in [
        TARGET_COLUMN,
        "LITHOLOGY_ENCODED",
        "GROUP_FOLD",
    ]:
        if excluded in numeric_columns:
            numeric_columns.remove(
                excluded
            )

    for column in numeric_columns:
        if (
            column in imputers
            and df[column].isna().any()
        ):
            df[column] = (
                imputers[column]
                .transform(
                    df[[column]]
                )
                .ravel()
            )

    df[numeric_columns] = (
        df[numeric_columns]
        .fillna(0)
        .replace(
            [np.inf, -np.inf],
            0,
        )
    )

    return df


def add_legacy_preprocess_features(
    dataframe,
):
    df = dataframe.copy()
    epsilon = 1e-10

    if {
        "NPHI",
        "RHOB",
    }.issubset(df.columns):
        df[
            "NPHI_RHOB_RATIO"
        ] = (
            df["NPHI"]
            / (
                df["RHOB"].abs()
                + epsilon
            )
        )

    if {
        "GR",
        "RHOB",
    }.issubset(df.columns):
        df[
            "GR_RHOB_RATIO"
        ] = (
            df["GR"]
            / (
                df["RHOB"].abs()
                + epsilon
            )
        )

    if {
        "DTC",
        "RHOB",
    }.issubset(df.columns):
        df[
            "DTC_RHOB_RATIO"
        ] = (
            df["DTC"]
            / (
                df["RHOB"].abs()
                + epsilon
            )
        )

    if "DEPTH_MD" in df.columns:
        depth_min = df[
            "DEPTH_MD"
        ].min()

        depth_max = df[
            "DEPTH_MD"
        ].max()

        depth_range = (
            depth_max
            - depth_min
        )

        if depth_range > 0:
            df[
                "DEPTH_NORMALIZED"
            ] = (
                (
                    df["DEPTH_MD"]
                    - depth_min
                )
                / depth_range
            )
        else:
            df[
                "DEPTH_NORMALIZED"
            ] = 0.0

    numeric_columns = (
        df.select_dtypes(
            include=[np.number]
        )
        .columns
    )

    df[numeric_columns] = (
        df[numeric_columns]
        .fillna(0)
        .replace(
            [np.inf, -np.inf],
            0,
        )
    )

    return df


def encode_target(
    dataframe,
):
    df = dataframe.copy()

    numeric_target = (
        pd.to_numeric(
            df[TARGET_COLUMN],
            errors="raise",
        )
        .astype(int)
    )

    unknown_codes = sorted(
        set(numeric_target)
        - set(CLASS_CODES)
    )

    if unknown_codes:
        raise ValueError(
            f"Unknown lithology codes: {unknown_codes}"
        )

    df[TARGET_COLUMN] = (
        numeric_target
    )

    df[
        "LITHOLOGY_ENCODED"
    ] = label_encoder.transform(
        numeric_target
    )

    return df


def apply_legacy_exact_resampling(
    dataframe,
):
    df = dataframe.copy()

    before_counts = (
        df[TARGET_COLUMN]
        .value_counts()
        .sort_index()
    )

    def drop_fraction(
        source_df,
        class_code,
        fraction,
        random_state,
    ):
        class_mask = (
            source_df[
                TARGET_COLUMN
            ]
            == class_code
        )

        class_count = int(
            class_mask.sum()
        )

        if class_count == 0:
            return source_df

        n_to_drop = int(
            class_count
            * fraction
        )

        indices_to_drop = (
            source_df.loc[
                class_mask
            ]
            .sample(
                n=n_to_drop,
                random_state=random_state,
            )
            .index
        )

        return source_df.drop(
            indices_to_drop
        )

    df = drop_fraction(
        df,
        class_code=65000,
        fraction=0.30,
        random_state=86,
    )

    df = drop_fraction(
        df,
        class_code=30000,
        fraction=0.20,
        random_state=86,
    )

    df = drop_fraction(
        df,
        class_code=70000,
        fraction=0.20,
        random_state=142,
    )

    df = drop_fraction(
        df,
        class_code=65030,
        fraction=0.20,
        random_state=142,
    )

    # رفتار واقعی کد مقاله قبلی حفظ می‌شود.
    basement_mask = (
        df[TARGET_COLUMN]
        == 93000
    )

    dolomite_mask = (
        df[TARGET_COLUMN]
        == 74000
    )

    dolomite_count = int(
        dolomite_mask.sum()
    )

    if (
        dolomite_count > 0
        and basement_mask.sum() > 0
    ):
        dolomite_target_count = int(
            dolomite_count
            * 1.5
        )

        n_to_add = (
            dolomite_target_count
            - dolomite_count
        )

        samples_to_add = (
            df.loc[
                basement_mask
            ]
            .sample(
                n=n_to_add,
                replace=True,
                random_state=42,
            )
        )

        df = pd.concat(
            [
                df,
                samples_to_add,
            ],
            ignore_index=True,
        )

    after_counts = (
        df[TARGET_COLUMN]
        .value_counts()
        .sort_index()
    )

    audit_rows = []

    for class_code in CLASS_CODES:
        before = int(
            before_counts.get(
                class_code,
                0,
            )
        )

        after = int(
            after_counts.get(
                class_code,
                0,
            )
        )

        audit_rows.append({
            "lithology_code": class_code,
            "lithology_name": (
                LITHOLOGY_NAMES[
                    class_code
                ]
            ),
            "before_count": before,
            "after_count": after,
            "change": after - before,
            "retained_fraction": (
                after / before
                if before > 0
                else np.nan
            ),
        })

    return (
        df,
        pd.DataFrame(
            audit_rows
        ),
    )

In [8]:
BASE_CAUSAL_FEATURES = [
    "GR",
    "NPHI",
    "RHOB",
    "PEF",
    "DTC",
    "SP",
]

def compute_legacy_causal_features(
    dataframe,
):
    df = dataframe.copy()

    available = [
        feature
        for feature in BASE_CAUSAL_FEATURES
        if feature in df.columns
    ]

    for feature_1, feature_2 in combinations(
        available,
        2,
    ):
        df[
            f"{feature_1}_{feature_2}_interaction"
        ] = (
            df[feature_1]
            * df[feature_2]
        )

    for feature in available:
        df[
            f"{feature}_squared"
        ] = (
            df[feature] ** 2
        )

        df[
            f"{feature}_log"
        ] = np.log(
            np.abs(
                df[feature]
            )
            + 1
        )

    return df


def add_legacy_weak_class_features(
    dataframe,
):
    df = dataframe.copy()
    epsilon = 1e-6

    def has_columns(
        columns,
    ):
        return all(
            column in df.columns
            for column in columns
        )

    # رفتار Legacy:
    # Train بدون WELL؛ Validation با WELL
    if has_columns(["GR"]):
        if "WELL" in df.columns:
            df["GR_grad"] = (
                df.groupby(
                    "WELL"
                )["GR"]
                .diff()
                .fillna(0)
            )

            df["GR_flatness"] = (
                df.groupby(
                    "WELL"
                )["GR"]
                .rolling(
                    window=5,
                    min_periods=1,
                )
                .std()
                .reset_index(
                    level=0,
                    drop=True,
                )
            )
        else:
            df["GR_grad"] = (
                df["GR"]
                .diff()
                .fillna(0)
            )

            df["GR_flatness"] = (
                df["GR"]
                .rolling(
                    5,
                    min_periods=1,
                )
                .std()
            )

    if has_columns(["DTC"]):
        if "WELL" in df.columns:
            df["DTC_grad"] = (
                df.groupby(
                    "WELL"
                )["DTC"]
                .diff()
                .fillna(0)
            )
        else:
            df["DTC_grad"] = (
                df["DTC"]
                .diff()
                .fillna(0)
            )

    if has_columns(["RDEP"]):
        if "WELL" in df.columns:
            df["RDEP_grad"] = (
                df.groupby(
                    "WELL"
                )["RDEP"]
                .diff()
                .fillna(0)
            )
        else:
            df["RDEP_grad"] = (
                df["RDEP"]
                .diff()
                .fillna(0)
            )

    if has_columns([
        "DTC",
        "RHOB",
    ]):
        df[
            "DTC_RHOB_DIFF"
        ] = (
            df["DTC"]
            - df["RHOB"]
        )

        df[
            "DTC_RHOB_RATIO2"
        ] = (
            df["DTC"]
            / (
                df["RHOB"]
                + epsilon
            )
        )

    if has_columns([
        "RDEP",
        "GR",
    ]):
        df[
            "RDEP_GR_RATIO"
        ] = (
            df["RDEP"]
            / (
                df["GR"]
                + epsilon
            )
        )

    if has_columns([
        "RHOB",
        "NPHI",
    ]):
        df[
            "RHOB_NPHI_DIFF2"
        ] = (
            df["RHOB"]
            - df["NPHI"]
        )

    if has_columns(["GR"]):
        gr_min = df["GR"].min()
        gr_max = df["GR"].max()

        df[
            "VSH_linear"
        ] = (
            (
                df["GR"]
                - gr_min
            )
            / (
                gr_max
                - gr_min
                + epsilon
            )
        )

        df[
            "VSH_Larionov"
        ] = (
            0.083
            * (
                2 ** (
                    3.7
                    * df[
                        "VSH_linear"
                    ]
                )
                - 1
            )
        )

        df[
            "VSH_Stieber"
        ] = (
            df["VSH_linear"]
            / (
                3
                - 2
                * df[
                    "VSH_linear"
                ]
                + epsilon
            )
        )

    if has_columns(["DTC"]):
        df["Vp"] = (
            1e6
            / (
                df["DTC"]
                + epsilon
            )
        )

    if has_columns([
        "RHOB",
        "Vp",
    ]):
        df[
            "Acoustic_Impedance"
        ] = (
            df["RHOB"]
            * df["Vp"]
        )

    if has_columns([
        "NPHI",
        "RHOB",
    ]):
        df[
            "Poisson_proxy"
        ] = (
            df["NPHI"]
            / (
                df["RHOB"]
                + epsilon
            )
        )

    if "NPHI" in df.columns:
        df["Brittleness"] = (
            1
            - df["NPHI"]
        )

    if has_columns([
        "Vp",
        "RHOB",
    ]):
        df[
            "Sand_indicator"
        ] = (
            df["Vp"]
            / (
                df["RHOB"]
                + epsilon
            )
        )

    if has_columns([
        "RHOB",
        "NPHI",
        "VSH_Larionov",
    ]):
        df[
            "Shale_indicator"
        ] = (
            df["RHOB"]
            * df["NPHI"]
            * df[
                "VSH_Larionov"
            ]
        )

    numeric_columns = (
        df.select_dtypes(
            include=[np.number]
        )
        .columns
    )

    df[numeric_columns] = (
        df[numeric_columns]
        .fillna(0)
        .replace(
            [np.inf, -np.inf],
            0,
        )
    )

    return df

In [9]:
def align_probabilities(
    raw_probabilities,
    trained_classes,
):
    aligned = np.zeros(
        (
            raw_probabilities.shape[0],
            len(CLASS_INDICES),
        ),
        dtype=np.float32,
    )

    for source_column, class_index in enumerate(
        trained_classes
    ):
        aligned[
            :,
            int(class_index),
        ] = raw_probabilities[
            :,
            source_column,
        ]

    row_sums = aligned.sum(
        axis=1,
        keepdims=True,
    )

    aligned = np.divide(
        aligned,
        row_sums,
        out=np.zeros_like(aligned),
        where=row_sums > 0,
    )

    return aligned


def calculate_metrics(
    y_true,
    y_pred,
    probabilities=None,
):
    present_labels = sorted(
        np.unique(y_true).tolist()
    )

    metrics = {
        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "macro_f1_present_classes": float(
            f1_score(
                y_true,
                y_pred,
                labels=present_labels,
                average="macro",
                zero_division=0,
            )
        ),
        "macro_f1_all_12_classes": float(
            f1_score(
                y_true,
                y_pred,
                labels=CLASS_INDICES,
                average="macro",
                zero_division=0,
            )
        ),
        "weighted_f1": float(
            f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0,
            )
        ),
        "macro_precision_present_classes": float(
            precision_score(
                y_true,
                y_pred,
                labels=present_labels,
                average="macro",
                zero_division=0,
            )
        ),
        "macro_recall_present_classes": float(
            recall_score(
                y_true,
                y_pred,
                labels=present_labels,
                average="macro",
                zero_division=0,
            )
        ),
    }

    if probabilities is not None:
        metrics["log_loss"] = float(
            log_loss(
                y_true,
                probabilities,
                labels=CLASS_INDICES,
            )
        )

    return metrics


def calculate_per_class_metrics(
    y_true,
    y_pred,
):
    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=CLASS_INDICES,
            zero_division=0,
        )
    )

    return pd.DataFrame({
        "class_index": CLASS_INDICES,
        "lithology_code": CLASS_CODES,
        "lithology_name": [
            CLASS_INDEX_TO_NAME[index]
            for index in CLASS_INDICES
        ],
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "support": support,
    })


def calculate_per_well_metrics(
    metadata,
    raw_probabilities,
    raw_predictions,
    refined_predictions,
):
    rows = []

    for well_name, well_indices in (
        metadata.groupby(
            "WELL"
        ).groups.items()
    ):
        positions = np.asarray(
            list(well_indices),
            dtype=int,
        )

        y_true = (
            metadata.loc[
                positions,
                "LITHOLOGY_ENCODED",
            ]
            .to_numpy(dtype=int)
        )

        probabilities = (
            raw_probabilities[
                positions
            ]
        )

        raw_pred = (
            raw_predictions[
                positions
            ]
        )

        refined_pred = (
            refined_predictions[
                positions
            ]
        )

        present_labels = sorted(
            np.unique(y_true).tolist()
        )

        rows.append({
            "WELL": well_name,
            "row_count": int(
                len(positions)
            ),
            "class_count": int(
                len(present_labels)
            ),
            "raw_accuracy": float(
                accuracy_score(
                    y_true,
                    raw_pred,
                )
            ),
            "refined_accuracy": float(
                accuracy_score(
                    y_true,
                    refined_pred,
                )
            ),
            "raw_macro_f1_present_classes": float(
                f1_score(
                    y_true,
                    raw_pred,
                    labels=present_labels,
                    average="macro",
                    zero_division=0,
                )
            ),
            "refined_macro_f1_present_classes": float(
                f1_score(
                    y_true,
                    refined_pred,
                    labels=present_labels,
                    average="macro",
                    zero_division=0,
                )
            ),
            "raw_log_loss": float(
                log_loss(
                    y_true,
                    probabilities,
                    labels=CLASS_INDICES,
                )
            ),
        })

    return pd.DataFrame(rows)

In [ ]:
all_fold_summaries = []
all_class_metrics = []
all_well_metrics = []
all_feature_importance = []
all_resampling_audits = []
run_manifest = []

for outer_fold in ACTIVE_OUTER_FOLDS:
    print("\n" + "=" * 110)
    print(f"OUTER FOLD {outer_fold}")
    print("=" * 110)

    run_name = (
        f"legacy_exact"
        f"__outer_fold_{outer_fold}"
    )

    fold_summary_path = (
        METRIC_DIR
        / f"{run_name}__summary.json"
    )

    class_metric_path = (
        METRIC_DIR
        / f"{run_name}__per_class.csv"
    )

    well_metric_path = (
        METRIC_DIR
        / f"{run_name}__per_well.csv"
    )

    prediction_path = (
        PREDICTION_DIR
        / f"{run_name}__oof.parquet"
    )

    importance_path = (
        IMPORTANCE_DIR
        / f"{run_name}.csv"
    )

    resampling_path = (
        RESAMPLING_DIR
        / f"{run_name}.csv"
    )

    scaler_path = (
        ARTIFACT_DIR
        / f"{run_name}__scaler.joblib"
    )

    imputer_path = (
        ARTIFACT_DIR
        / f"{run_name}__imputers.joblib"
    )

    model_path = (
        ARTIFACT_DIR
        / f"{run_name}__random_forest.joblib"
    )

    completed_files = [
        fold_summary_path,
        class_metric_path,
        well_metric_path,
        prediction_path,
        importance_path,
        resampling_path,
    ]

    if (
        SKIP_COMPLETED_FOLDS
        and all(
            path.exists()
            for path in completed_files
        )
    ):
        print(
            "Completed artifacts found. Skipping fold."
        )

        with open(
            fold_summary_path,
            "r",
            encoding="utf-8",
        ) as file:
            all_fold_summaries.append(
                json.load(file)
            )

        class_df = pd.read_csv(
            class_metric_path
        )

        well_df = pd.read_csv(
            well_metric_path
        )

        importance_df = pd.read_csv(
            importance_path
        )

        resampling_df = pd.read_csv(
            resampling_path
        )

        all_class_metrics.append(
            class_df
        )

        all_well_metrics.append(
            well_df
        )

        all_feature_importance.append(
            importance_df
        )

        all_resampling_audits.append(
            resampling_df
        )

        continue

    validation_mask = (
        train_raw[
            "GROUP_FOLD"
        ].to_numpy()
        == outer_fold
    )

    training_mask = (
        ~validation_mask
    )

    raw_outer_train = (
        train_raw.loc[
            training_mask
        ]
        .drop(
            columns=["GROUP_FOLD"]
        )
        .copy()
        .reset_index(drop=True)
    )

    raw_outer_validation = (
        train_raw.loc[
            validation_mask
        ]
        .drop(
            columns=["GROUP_FOLD"]
        )
        .copy()
        .reset_index(drop=True)
    )

    train_wells = set(
        raw_outer_train[
            "WELL"
        ].dropna()
    )

    validation_wells = set(
        raw_outer_validation[
            "WELL"
        ].dropna()
    )

    overlap = (
        train_wells
        & validation_wells
    )

    if overlap:
        raise RuntimeError(
            f"WELL leakage detected: {sorted(overlap)}"
        )

    print(
        "Raw outer train:",
        raw_outer_train.shape,
        "| wells:",
        len(train_wells),
    )

    print(
        "Raw outer validation:",
        raw_outer_validation.shape,
        "| wells:",
        len(validation_wells),
    )

    # ---------------------------------------------------
    # Train preprocessing
    # ---------------------------------------------------

    (
        train_imputed,
        train_imputers,
    ) = fit_and_apply_train_imputers(
        raw_outer_train
    )

    train_preprocessed = (
        add_legacy_preprocess_features(
            train_imputed
        )
    )

    (
        train_resampled,
        resampling_audit,
    ) = apply_legacy_exact_resampling(
        train_preprocessed
    )

    train_resampled = encode_target(
        train_resampled
    )

    train_causal = (
        compute_legacy_causal_features(
            train_resampled
        )
    )

    exclude_columns = {
        "WELL",
        TARGET_COLUMN,
        CONFIDENCE_COLUMN,
        "LITHOLOGY_ENCODED",
        "GROUP",
        "FORMATION",
    }

    candidate_numeric_features = [
        column
        for column in train_causal.columns
        if (
            column not in exclude_columns
            and pd.api.types.is_numeric_dtype(
                train_causal[column]
            )
        )
    ]

    
    train_numeric_only = (
        train_causal[
            candidate_numeric_features
        ]
        .copy()
    )

    train_with_weak = (
        add_legacy_weak_class_features(
            train_numeric_only
        )
    )

    missing_train_features = [
        feature
        for feature in set(
            LEGACY_FEATURE_COLUMNS
        )
        if feature
        not in train_with_weak.columns
    ]

    if missing_train_features:
        raise RuntimeError(
            f"Missing Train features: {missing_train_features}"
        )

    X_train = (
        train_with_weak[
            LEGACY_FEATURE_COLUMNS
        ]
        .copy()
    )

    y_train = (
        train_causal[
            "LITHOLOGY_ENCODED"
        ]
        .astype(int)
        .to_numpy()
    )

    # ---------------------------------------------------
    # Validation preprocessing
    # ---------------------------------------------------

    validation_imputed = (
        apply_train_imputers_to_validation(
            raw_outer_validation,
            imputers=train_imputers,
        )
    )

   
    validation_preprocessed = (
        add_legacy_preprocess_features(
            validation_imputed
        )
    )

    validation_preprocessed = (
        encode_target(
            validation_preprocessed
        )
    )

    validation_causal = (
        compute_legacy_causal_features(
            validation_preprocessed
        )
    )

   
    validation_with_weak = (
        add_legacy_weak_class_features(
            validation_causal
        )
    )

    missing_validation_features = [
        feature
        for feature in set(
            LEGACY_FEATURE_COLUMNS
        )
        if feature
        not in validation_with_weak.columns
    ]

    if missing_validation_features:
        raise RuntimeError(
            "Missing Validation features: "
            f"{missing_validation_features}"
        )

    X_validation = (
        validation_with_weak[
            LEGACY_FEATURE_COLUMNS
        ]
        .copy()
    )

    y_validation = (
        validation_with_weak[
            "LITHOLOGY_ENCODED"
        ]
        .astype(int)
        .to_numpy()
    )

    validation_metadata = (
        validation_with_weak[
            [
                "WELL",
                "DEPTH_MD",
                TARGET_COLUMN,
                "LITHOLOGY_ENCODED",
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )

    # ---------------------------------------------------
    # Scale + Random Forest
    # ---------------------------------------------------

    scaler = StandardScaler()

    X_train_scaled = (
        scaler.fit_transform(
            X_train
        )
    )

    X_validation_scaled = (
        scaler.transform(
            X_validation
        )
    )

    model = RandomForestClassifier(
        n_estimators=RF_N_ESTIMATORS,
        max_depth=RF_MAX_DEPTH,
        min_samples_split=(
            RF_MIN_SAMPLES_SPLIT
        ),
        min_samples_leaf=(
            RF_MIN_SAMPLES_LEAF
        ),
        random_state=(
            RANDOM_STATE_MODEL
        ),
        n_jobs=RF_N_JOBS,
        verbose=1,
    )

    fit_start = time.time()

    model.fit(
        X_train_scaled,
        y_train,
    )

    fit_seconds = (
        time.time()
        - fit_start
    )

    raw_probability_partial = (
        model.predict_proba(
            X_validation_scaled
        )
    )

    raw_probabilities = (
        align_probabilities(
            raw_probability_partial,
            model.classes_,
        )
    )

    raw_predictions = (
        raw_probabilities.argmax(
            axis=1
        )
        .astype(int)
    )

    # ---------------------------------------------------
    # Legacy refinement
    # ---------------------------------------------------

    probability_order = np.argsort(
        raw_probabilities,
        axis=1,
    )[:, ::-1]

    second_class = (
        probability_order[:, 1]
    )

    refined_predictions = (
        raw_predictions.copy()
    )

    refinement_mask = (
        (second_class == CHALK_INDEX)
        & (
            raw_predictions
            != CHALK_INDEX
        )
    )

    refined_predictions[
        refinement_mask
    ] = CHALK_INDEX

    # ---------------------------------------------------
    # Metrics
    # ---------------------------------------------------

    raw_metrics = calculate_metrics(
        y_validation,
        raw_predictions,
        probabilities=raw_probabilities,
    )

    refined_metrics = (
        calculate_metrics(
            y_validation,
            refined_predictions,
        )
    )

    per_class_raw = (
        calculate_per_class_metrics(
            y_validation,
            raw_predictions,
        )
    )

    per_class_raw[
        "prediction_variant"
    ] = "raw"

    per_class_refined = (
        calculate_per_class_metrics(
            y_validation,
            refined_predictions,
        )
    )

    per_class_refined[
        "prediction_variant"
    ] = "legacy_refined"

    per_class = pd.concat(
        [
            per_class_raw,
            per_class_refined,
        ],
        ignore_index=True,
    )

    per_class[
        "outer_fold"
    ] = outer_fold

    per_well = (
        calculate_per_well_metrics(
            validation_metadata,
            raw_probabilities,
            raw_predictions,
            refined_predictions,
        )
    )

    per_well[
        "outer_fold"
    ] = outer_fold

    train_classes = sorted(
        np.unique(
            y_train
        ).tolist()
    )

    validation_classes = sorted(
        np.unique(
            y_validation
        ).tolist()
    )

    classes_absent_from_train = sorted(
        set(validation_classes)
        - set(train_classes)
    )

    fold_summary = {
        "run_mode": RUN_MODE,
        "outer_fold": int(
            outer_fold
        ),
        "train_rows_raw": int(
            len(raw_outer_train)
        ),
        "train_rows_after_legacy_resampling": int(
            len(X_train)
        ),
        "validation_rows": int(
            len(X_validation)
        ),
        "train_wells": int(
            len(train_wells)
        ),
        "validation_wells": int(
            len(validation_wells)
        ),
        "well_overlap_count": 0,
        "train_class_indices": (
            train_classes
        ),
        "validation_class_indices": (
            validation_classes
        ),
        "validation_classes_absent_from_train": (
            classes_absent_from_train
        ),
        "raw_metrics": raw_metrics,
        "legacy_refined_metrics": (
            refined_metrics
        ),
        "refinement_changed_rows": int(
            refinement_mask.sum()
        ),
        "refinement_changed_fraction": float(
            refinement_mask.mean()
        ),
        "fit_seconds": float(
            fit_seconds
        ),
    }

    with open(
        fold_summary_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            fold_summary,
            file,
            ensure_ascii=False,
            indent=2,
        )

    per_class.to_csv(
        class_metric_path,
        index=False,
    )

    per_well.to_csv(
        well_metric_path,
        index=False,
    )

    resampling_audit[
        "outer_fold"
    ] = outer_fold

    resampling_audit.to_csv(
        resampling_path,
        index=False,
    )

    feature_importance = pd.DataFrame({
        "feature_position": list(
            range(
                len(
                    LEGACY_FEATURE_COLUMNS
                )
            )
        ),
        "feature": (
            LEGACY_FEATURE_COLUMNS
        ),
        "importance": (
            model.feature_importances_
        ),
        "outer_fold": outer_fold,
    }).sort_values(
        "importance",
        ascending=False,
    )

    feature_importance.to_csv(
        importance_path,
        index=False,
    )

    if SAVE_OOF_PROBABILITIES:
        prediction_output = (
            validation_metadata.copy()
        )

        prediction_output[
            "outer_fold"
        ] = outer_fold

        prediction_output[
            "raw_prediction_index"
        ] = raw_predictions

        prediction_output[
            "legacy_refined_prediction_index"
        ] = refined_predictions

        prediction_output[
            "legacy_refinement_modified"
        ] = refinement_mask

        for class_index, class_code in enumerate(
            CLASS_CODES
        ):
            prediction_output[
                f"prob_{class_code}"
            ] = raw_probabilities[
                :,
                class_index,
            ]

        prediction_output.to_parquet(
            prediction_path,
            index=False,
        )

    if SAVE_PREPROCESSORS:
        joblib.dump(
            scaler,
            scaler_path,
            compress=3,
        )

        joblib.dump(
            train_imputers,
            imputer_path,
            compress=3,
        )

    if SAVE_MODELS:
        print(
            "Saving large Random Forest model ..."
        )

        joblib.dump(
            model,
            model_path,
            compress=3,
        )

    all_fold_summaries.append(
        fold_summary
    )

    all_class_metrics.append(
        per_class
    )

    all_well_metrics.append(
        per_well
    )

    all_feature_importance.append(
        feature_importance
    )

    all_resampling_audits.append(
        resampling_audit
    )

    run_manifest.append({
        "run_name": run_name,
        "outer_fold": int(
            outer_fold
        ),
        "status": "completed",
        "prediction_path": str(
            prediction_path
        ),
        "model_saved": bool(
            SAVE_MODELS
        ),
        "model_path": (
            str(model_path)
            if SAVE_MODELS
            else None
        ),
    })

    print(
        json.dumps(
            fold_summary,
            indent=2,
        )
    )

    del (
        raw_outer_train,
        raw_outer_validation,
        train_imputed,
        train_preprocessed,
        train_resampled,
        train_causal,
        train_numeric_only,
        train_with_weak,
        X_train,
        y_train,
        validation_imputed,
        validation_preprocessed,
        validation_causal,
        validation_with_weak,
        X_validation,
        y_validation,
        X_train_scaled,
        X_validation_scaled,
        raw_probability_partial,
        raw_probabilities,
        raw_predictions,
        refined_predictions,
        probability_order,
        model,
        scaler,
    )

    gc.collect()


OUTER FOLD 1
Raw outer train: (937837, 29) | wells: 78
Raw outer validation: (232674, 29) | wells: 20


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done  30 tasks      | elapsed:   19.2s
[Parallel(n_jobs=-1)]: Done 180 tasks      | elapsed:  1.8min
[Parallel(n_jobs=-1)]: Done 430 tasks      | elapsed:  4.4min
[Parallel(n_jobs=-1)]: Done 780 tasks      | elapsed:  7.9min
[Parallel(n_jobs=-1)]: Done 800 out of 800 | elapsed:  8.1min finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.2s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    0.6s
[Parallel(n_jobs=10)]: Done 430 tasks      | elapsed:    1.4s
[Parallel(n_jobs=10)]: Done 780 tasks      | elapsed:    2.6s
[Parallel(n_jobs=10)]: Done 800 out of 800 | elapsed:    2.6s finished


Saving large Random Forest model ...
{
  "run_mode": "full",
  "outer_fold": 1,
  "train_rows_raw": 937837,
  "train_rows_after_legacy_resampling": 704041,
  "validation_rows": 232674,
  "train_wells": 78,
  "validation_wells": 20,
  "well_overlap_count": 0,
  "train_class_indices": [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11
  ],
  "validation_class_indices": [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    11
  ],
  "validation_classes_absent_from_train": [],
  "raw_metrics": {
    "accuracy": 0.7382217179401223,
    "balanced_accuracy": 0.3644790027139606,
    "macro_f1_present_classes": 0.38182235100996675,
    "macro_f1_all_12_classes": 0.3500038217591362,
    "weighted_f1": 0.6860069292366159,
    "macro_precision_present_classes": 0.5394161037264774,
    "macro_recall_present_classes": 0.3644790027139606,
    "log_loss": 0.8107490606144344
  },
  "legacy_refined_metrics": {
    "accuracy": 0.7348135159063754,


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done  30 tasks      | elapsed:   19.5s
[Parallel(n_jobs=-1)]: Done 180 tasks      | elapsed:  1.8min
[Parallel(n_jobs=-1)]: Done 430 tasks      | elapsed:  4.2min
[Parallel(n_jobs=-1)]: Done 780 tasks      | elapsed:  7.8min
[Parallel(n_jobs=-1)]: Done 800 out of 800 | elapsed:  8.0min finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.1s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    0.6s
[Parallel(n_jobs=10)]: Done 430 tasks      | elapsed:    1.4s
[Parallel(n_jobs=10)]: Done 780 tasks      | elapsed:    2.5s
[Parallel(n_jobs=10)]: Done 800 out of 800 | elapsed:    2.6s finished


Saving large Random Forest model ...
{
  "run_mode": "full",
  "outer_fold": 2,
  "train_rows_raw": 934306,
  "train_rows_after_legacy_resampling": 701915,
  "validation_rows": 236205,
  "train_wells": 78,
  "validation_wells": 20,
  "well_overlap_count": 0,
  "train_class_indices": [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    11
  ],
  "validation_class_indices": [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    9,
    10,
    11
  ],
  "validation_classes_absent_from_train": [
    10
  ],
  "raw_metrics": {
    "accuracy": 0.6943333121652802,
    "balanced_accuracy": 0.2907472349791575,
    "macro_f1_present_classes": 0.31149549841947893,
    "macro_f1_all_12_classes": 0.2855375402178557,
    "weighted_f1": 0.6467969234662743,
    "macro_precision_present_classes": 0.5413525958100813,
    "macro_recall_present_classes": 0.2907472349791575,
    "log_loss": 0.8580258021871296
  },
  "legacy_refined_metrics": {
    "accuracy": 0.698389111153447

...

Saving large Random Forest model ...
{
  "run_mode": "full",
  "outer_fold": 3,
  "train_rows_raw": 939933,
  "train_rows_after_legacy_resampling": 707034,
  "validation_rows": 230578,
  "train_wells": 77,
  "validation_wells": 21,
  "well_overlap_count": 0,
  "train_class_indices": [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11
  ],
  "validation_class_indices": [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    9,
    11
  ],
  "validation_classes_absent_from_train": [],
  "raw_metrics": {
    "accuracy": 0.7293020149363773,
    "balanced_accuracy": 0.33731479033515865,
    "macro_f1_present_classes": 0.3642444620721042,
    "macro_f1_all_12_classes": 0.3035370517267535,
    "weighted_f1": 0.6826477960216276,
    "macro_precision_present_classes": 0.43685923136180166,
    "macro_recall_present_classes": 0.33731479033515865,
    "log_loss": 0.8630779969061956
  },
  "legacy_refined_metrics": {
    "accuracy": 0.7233387400359098,
    "

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done  30 tasks      | elapsed:   17.7s
[Parallel(n_jobs=-1)]: Done 180 tasks      | elapsed:  1.9min
[Parallel(n_jobs=-1)]: Done 430 tasks      | elapsed:  4.5min
[Parallel(n_jobs=-1)]: Done 780 tasks      | elapsed:  8.2min
[Parallel(n_jobs=-1)]: Done 800 out of 800 | elapsed:  8.4min finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.1s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    0.6s
[Parallel(n_jobs=10)]: Done 430 tasks      | elapsed:    1.5s
[Parallel(n_jobs=10)]: Done 780 tasks      | elapsed:    2.7s
[Parallel(n_jobs=10)]: Done 800 out of 800 | elapsed:    2.7s finished


Saving large Random Forest model ...
{
  "run_mode": "full",
  "outer_fold": 4,
  "train_rows_raw": 933379,
  "train_rows_after_legacy_resampling": 700801,
  "validation_rows": 237132,
  "train_wells": 81,
  "validation_wells": 17,
  "well_overlap_count": 0,
  "train_class_indices": [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11
  ],
  "validation_class_indices": [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    11
  ],
  "validation_classes_absent_from_train": [],
  "raw_metrics": {
    "accuracy": 0.6921292782079179,
    "balanced_accuracy": 0.3227880475827966,
    "macro_f1_present_classes": 0.3455281028901002,
    "macro_f1_all_12_classes": 0.3167340943159252,
    "weighted_f1": 0.6647193265603663,
    "macro_precision_present_classes": 0.4308096047918518,
    "macro_recall_present_classes": 0.3227880475827966,
    "log_loss": 0.8844158492751836
  },
  "legacy_refined_metrics": {
    "accuracy": 0.6961607880842737,
 

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done  30 tasks      | elapsed:   18.6s
[Parallel(n_jobs=-1)]: Done 180 tasks      | elapsed:  1.9min
[Parallel(n_jobs=-1)]: Done 430 tasks      | elapsed:  4.6min
[Parallel(n_jobs=-1)]: Done 780 tasks      | elapsed:  8.5min
[Parallel(n_jobs=-1)]: Done 800 out of 800 | elapsed:  8.7min finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.1s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    0.6s
[Parallel(n_jobs=10)]: Done 430 tasks      | elapsed:    1.4s
[Parallel(n_jobs=10)]: Done 780 tasks      | elapsed:    2.6s
[Parallel(n_jobs=10)]: Done 800 out of 800 | elapsed:    2.6s finished


Saving large Random Forest model ...
{
  "run_mode": "full",
  "outer_fold": 5,
  "train_rows_raw": 936589,
  "train_rows_after_legacy_resampling": 705424,
  "validation_rows": 233922,
  "train_wells": 78,
  "validation_wells": 20,
  "well_overlap_count": 0,
  "train_class_indices": [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11
  ],
  "validation_class_indices": [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    11
  ],
  "validation_classes_absent_from_train": [],
  "raw_metrics": {
    "accuracy": 0.76918801993827,
    "balanced_accuracy": 0.32076137619938566,
    "macro_f1_present_classes": 0.35171602459598833,
    "macro_f1_all_12_classes": 0.322406355879656,
    "weighted_f1": 0.741000737610983,
    "macro_precision_present_classes": 0.4983237443717438,
    "macro_recall_present_classes": 0.32076137619938566,
    "log_loss": 0.7331370074179104
  },
  "legacy_refined_metrics": {
    "accuracy": 0.7629808226673849,
  

In [11]:
fold_rows = []

for summary in all_fold_summaries:
    raw = summary[
        "raw_metrics"
    ]

    refined = summary[
        "legacy_refined_metrics"
    ]

    fold_rows.append({
        "run_mode": summary[
            "run_mode"
        ],
        "outer_fold": summary[
            "outer_fold"
        ],
        "train_rows_raw": summary[
            "train_rows_raw"
        ],
        "train_rows_after_legacy_resampling": summary[
            "train_rows_after_legacy_resampling"
        ],
        "validation_rows": summary[
            "validation_rows"
        ],
        "train_wells": summary[
            "train_wells"
        ],
        "validation_wells": summary[
            "validation_wells"
        ],
        "raw_accuracy": raw[
            "accuracy"
        ],
        "raw_balanced_accuracy": raw[
            "balanced_accuracy"
        ],
        "raw_macro_f1_present_classes": raw[
            "macro_f1_present_classes"
        ],
        "raw_macro_f1_all_12_classes": raw[
            "macro_f1_all_12_classes"
        ],
        "raw_weighted_f1": raw[
            "weighted_f1"
        ],
        "raw_log_loss": raw[
            "log_loss"
        ],
        "refined_accuracy": refined[
            "accuracy"
        ],
        "refined_balanced_accuracy": refined[
            "balanced_accuracy"
        ],
        "refined_macro_f1_present_classes": refined[
            "macro_f1_present_classes"
        ],
        "refined_macro_f1_all_12_classes": refined[
            "macro_f1_all_12_classes"
        ],
        "refined_weighted_f1": refined[
            "weighted_f1"
        ],
        "refinement_changed_rows": summary[
            "refinement_changed_rows"
        ],
        "refinement_changed_fraction": summary[
            "refinement_changed_fraction"
        ],
        "fit_seconds": summary[
            "fit_seconds"
        ],
        "validation_classes_absent_from_train": json.dumps(
            summary[
                "validation_classes_absent_from_train"
            ]
        ),
    })

fold_metrics_df = pd.DataFrame(
    fold_rows
)

display(fold_metrics_df)

summary_metric_columns = [
    column
    for column in fold_metrics_df.columns
    if (
        column.startswith("raw_")
        or column.startswith("refined_")
        or column
        in [
            "refinement_changed_fraction",
            "fit_seconds",
        ]
    )
]

grouped_summary = (
    fold_metrics_df[
        summary_metric_columns
    ]
    .agg(
        [
            "mean",
            "std",
            "min",
            "max",
        ]
    )
    .T
    .reset_index()
    .rename(
        columns={
            "index": "metric"
        }
    )
)

display(grouped_summary)

,run_mode,outer_fold,train_rows_raw,train_rows_after_legacy_resampling,validation_rows,train_wells,validation_wells,raw_accuracy,raw_balanced_accuracy,raw_macro_f1_present_classes,raw_macro_f1_all_12_classes,raw_weighted_f1,raw_log_loss,refined_accuracy,refined_balanced_accuracy,refined_macro_f1_present_classes,refined_macro_f1_all_12_classes,refined_weighted_f1,refinement_changed_rows,refinement_changed_fraction,fit_seconds,validation_classes_absent_from_train
0,full,1,937837,704041,232674,78,20,0.738222,0.364479,0.381822,0.350004,0.686007,0.810749,0.734814,0.398314,0.386853,0.354615,0.682429,3832,0.016469,484.744010,[]
1,full,2,934306,701915,236205,78,20,0.694333,0.290747,0.311495,0.285538,0.646797,0.858026,0.698389,0.358036,0.373622,0.342487,0.652728,3312,0.014022,479.838695,[10]
2,full,3,939933,707034,230578,77,21,0.729302,0.337315,0.364244,0.303537,0.682648,0.863078,0.723339,0.328957,0.358074,0.298395,0.678166,1443,0.006258,521.671443,[]
3,full,4,933379,700801,237132,81,17,0.692129,0.322788,0.345528,0.316734,0.664719,0.884416,0.696161,0.357005,0.395669,0.362697,0.671440,1081,0.004559,504.327297,[]
4,full,5,936589,705424,233922,78,20,0.769188,0.320761,0.351716,0.322406,0.741001,0.733137,0.762981,0.327421,0.356790,0.327057,0.736623,2096,0.008960,524.140726,[]


,metric,mean,std,min,max
0,raw_accuracy,0.724635,0.032273,0.692129,0.769188
1,raw_balanced_accuracy,0.327218,0.026841,0.290747,0.364479
2,raw_macro_f1_present_classes,0.350961,0.026060,0.311495,0.381822
3,raw_macro_f1_all_12_classes,0.315644,0.023877,0.285538,0.350004
4,raw_weighted_f1,0.684234,0.035392,0.646797,0.741001
5,raw_log_loss,0.829881,0.060402,0.733137,0.884416
6,refined_accuracy,0.723137,0.027677,0.696161,0.762981
7,refined_balanced_accuracy,0.353947,0.028821,0.327421,0.398314
8,refined_macro_f1_present_classes,0.374202,0.017209,0.356790,0.395669
9,refined_macro_f1_all_12_classes,0.337050,0.025446,0.298395,0.362697


In [12]:
per_class_all = (
    pd.concat(
        all_class_metrics,
        ignore_index=True,
    )
    if all_class_metrics
    else pd.DataFrame()
)

per_well_all = (
    pd.concat(
        all_well_metrics,
        ignore_index=True,
    )
    if all_well_metrics
    else pd.DataFrame()
)

feature_importance_all = (
    pd.concat(
        all_feature_importance,
        ignore_index=True,
    )
    if all_feature_importance
    else pd.DataFrame()
)

resampling_all = (
    pd.concat(
        all_resampling_audits,
        ignore_index=True,
    )
    if all_resampling_audits
    else pd.DataFrame()
)

if not feature_importance_all.empty:
    mean_feature_importance = (
        feature_importance_all
        .groupby(
            [
                "feature_position",
                "feature",
            ],
            as_index=False,
        )
        .agg(
            mean_importance=(
                "importance",
                "mean",
            ),
            std_importance=(
                "importance",
                "std",
            ),
            fold_count=(
                "outer_fold",
                "nunique",
            ),
        )
        .sort_values(
            "mean_importance",
            ascending=False,
        )
    )

    display(
        mean_feature_importance.head(20)
    )
else:
    mean_feature_importance = pd.DataFrame()

,feature_position,feature,mean_importance,std_importance,fold_count
2,2,Y_LOC,0.074647,0.006792,5
0,0,GR_NPHI_interaction,0.070531,0.003295,5
1,1,GR_DTC_interaction,0.065317,0.002856,5
25,25,Shale_indicator,0.061702,0.002233,5
5,5,X_LOC,0.061067,0.002887,5
16,16,X_LOC,0.060902,0.002447,5
8,8,Z_LOC,0.053046,0.000808,5
11,11,DEPTH_MD,0.051359,0.001244,5
10,10,DEPTH_NORMALIZED,0.051273,0.000734,5
7,7,CALI,0.044506,0.000998,5


In [13]:
oof_frames = []

for outer_fold in ACTIVE_OUTER_FOLDS:
    path = (
        PREDICTION_DIR
        / (
            "legacy_exact"
            f"__outer_fold_{outer_fold}"
            "__oof.parquet"
        )
    )

    if path.exists():
        oof_frames.append(
            pd.read_parquet(
                path
            )
        )

if oof_frames:
    oof_predictions = pd.concat(
        oof_frames,
        ignore_index=True,
    )

    y_oof = (
        oof_predictions[
            "LITHOLOGY_ENCODED"
        ]
        .to_numpy(dtype=int)
    )

    raw_oof_predictions = (
        oof_predictions[
            "raw_prediction_index"
        ]
        .to_numpy(dtype=int)
    )

    refined_oof_predictions = (
        oof_predictions[
            "legacy_refined_prediction_index"
        ]
        .to_numpy(dtype=int)
    )

    probability_columns = [
        f"prob_{class_code}"
        for class_code in CLASS_CODES
    ]

    oof_probabilities = (
        oof_predictions[
            probability_columns
        ]
        .to_numpy(
            dtype=np.float32
        )
    )

    oof_raw_metrics = calculate_metrics(
        y_oof,
        raw_oof_predictions,
        probabilities=oof_probabilities,
    )

    oof_refined_metrics = (
        calculate_metrics(
            y_oof,
            refined_oof_predictions,
        )
    )

    oof_summary = pd.DataFrame([
        {
            "prediction_variant": "raw",
            **oof_raw_metrics,
        },
        {
            "prediction_variant": (
                "legacy_refined"
            ),
            **oof_refined_metrics,
        },
    ])

    display(oof_summary)

else:
    oof_predictions = pd.DataFrame()
    oof_summary = pd.DataFrame()
    print("No OOF prediction files were found.")

,prediction_variant,accuracy,balanced_accuracy,macro_f1_present_classes,macro_f1_all_12_classes,weighted_f1,macro_precision_present_classes,macro_recall_present_classes,log_loss
0,raw,0.724459,0.310379,0.342907,0.342907,0.685022,0.478220,0.310379,0.830011
1,legacy_refined,0.723001,0.339798,0.367302,0.367302,0.685164,0.444439,0.339798,NaN


In [14]:
legacy_reference = {
    "observed_raw_accuracy": (
        0.797944394061946
    ),
    "observed_refined_accuracy": (
        0.8077648961984363
    ),
}

if LEGACY_REFERENCE_GATE_PATH.exists():
    with open(
        LEGACY_REFERENCE_GATE_PATH,
        "r",
        encoding="utf-8",
    ) as file:
        gate = json.load(file)

    legacy_reference[
        "observed_raw_accuracy"
    ] = gate[
        "observed_raw_accuracy"
    ]

    legacy_reference[
        "observed_refined_accuracy"
    ] = gate[
        "observed_refined_accuracy"
    ]

comparison_rows = []

if not oof_summary.empty:
    raw_oof_row = (
        oof_summary.loc[
            oof_summary[
                "prediction_variant"
            ]
            == "raw"
        ]
        .iloc[0]
    )

    refined_oof_row = (
        oof_summary.loc[
            oof_summary[
                "prediction_variant"
            ]
            == "legacy_refined"
        ]
        .iloc[0]
    )

    comparison_rows = [
        {
            "evaluation_protocol": (
                "Official 10-well test"
            ),
            "prediction_variant": (
                "raw"
            ),
            "accuracy": (
                legacy_reference[
                    "observed_raw_accuracy"
                ]
            ),
        },
        {
            "evaluation_protocol": (
                "Official 10-well test"
            ),
            "prediction_variant": (
                "legacy_refined"
            ),
            "accuracy": (
                legacy_reference[
                    "observed_refined_accuracy"
                ]
            ),
        },
        {
            "evaluation_protocol": (
                f"Grouped WELL OOF ({RUN_MODE})"
            ),
            "prediction_variant": (
                "raw"
            ),
            "accuracy": float(
                raw_oof_row[
                    "accuracy"
                ]
            ),
        },
        {
            "evaluation_protocol": (
                f"Grouped WELL OOF ({RUN_MODE})"
            ),
            "prediction_variant": (
                "legacy_refined"
            ),
            "accuracy": float(
                refined_oof_row[
                    "accuracy"
                ]
            ),
        },
    ]

protocol_comparison = pd.DataFrame(
    comparison_rows
)

display(protocol_comparison)

,evaluation_protocol,prediction_variant,accuracy
0,Official 10-well test,raw,0.797944
1,Official 10-well test,legacy_refined,0.807765
2,Grouped WELL OOF (full),raw,0.724459
3,Grouped WELL OOF (full),legacy_refined,0.723001


In [ ]:

fold_assignment_summary = (
    train_raw
    .groupby("GROUP_FOLD")
    .agg(
        row_count=(
            "WELL",
            "size",
        ),
        well_count=(
            "WELL",
            "nunique",
        ),
    )
    .reset_index()
)



class_mapping.to_csv(
    OUTPUT_DIR
    / "00_class_mapping.csv",
    index=False,
)

fold_assignment_summary.to_csv(
    OUTPUT_DIR
    / "01_fold_assignment_summary.csv",
    index=False,
)

fold_metrics_df.to_csv(
    OUTPUT_DIR
    / "02_fold_metrics.csv",
    index=False,
)

grouped_summary.to_csv(
    OUTPUT_DIR
    / "03_grouped_metric_summary.csv",
    index=False,
)


# ------------------------------------------------------------

if not per_class_all.empty:
    per_class_all.to_csv(
        OUTPUT_DIR
        / "04_per_class_metrics.csv",
        index=False,
    )

if not per_well_all.empty:
    per_well_all.to_csv(
        OUTPUT_DIR
        / "05_per_well_metrics.csv",
        index=False,
    )

# ------------------------------------------------------------
# Feature importance
# ------------------------------------------------------------

if not feature_importance_all.empty:
    feature_importance_all.to_csv(
        OUTPUT_DIR
        / "06_feature_importance_by_fold.csv",
        index=False,
    )

if not mean_feature_importance.empty:
    mean_feature_importance.to_csv(
        OUTPUT_DIR
        / "07_mean_feature_importance.csv",
        index=False,
    )

# ------------------------------------------------------------
# Resampling audit
# ------------------------------------------------------------

if not resampling_all.empty:
    resampling_all.to_csv(
        OUTPUT_DIR
        / "08_resampling_audit_all_folds.csv",
        index=False,
    )

# ------------------------------------------------------------
# OOF results
# ------------------------------------------------------------

if not oof_summary.empty:
    oof_summary.to_csv(
        OUTPUT_DIR
        / "09_oof_summary.csv",
        index=False,
    )

# ------------------------------------------------------------
# مقایسه با Test رسمی مقاله قبلی
# ------------------------------------------------------------

if comparison_rows:
    protocol_comparison.to_csv(
        OUTPUT_DIR
        / "10_official_test_vs_grouped_oof.csv",
        index=False,
    )

# ------------------------------------------------------------
# Run manifest
# ------------------------------------------------------------

pd.DataFrame(
    run_manifest
).to_csv(
    OUTPUT_DIR
    / "11_run_manifest.csv",
    index=False,
)

# ------------------------------------------------------------
# ------------------------------------------------------------

run_configuration = {
    "stage": (
        "02B_legacy_RF_grouped_well_CV"
    ),
    "run_mode": RUN_MODE,
    "official_test_used": False,

    "outer_split": {
        "group": "WELL",
        "n_folds_total": 5,
        "active_folds": (
            ACTIVE_OUTER_FOLDS
        ),
        "assignment_path": str(
            FOLD_ASSIGNMENT_PATH
        ),
    },

    "legacy_model_unchanged": True,

    "random_forest": {
        "n_estimators": (
            RF_N_ESTIMATORS
        ),
        "max_depth": (
            RF_MAX_DEPTH
        ),
        "min_samples_split": (
            RF_MIN_SAMPLES_SPLIT
        ),
        "min_samples_leaf": (
            RF_MIN_SAMPLES_LEAF
        ),
        "random_state": (
            RANDOM_STATE_MODEL
        ),
        "n_jobs": (
            RF_N_JOBS
        ),
    },

    "feature_column_count": 26,
    "unique_feature_name_count": 25,

    "duplicate_feature_names": [
        "X_LOC"
    ],

    "legacy_behavior_preserved": {
        "validation_depth_normalization_uses_validation_min_max": True,
        "train_weak_features_without_WELL": True,
        "validation_weak_features_with_WELL": True,
        "basement_added_in_dolomite_branch": True,
        "refinement_condition": (
            "second-ranked class is Chalk"
        ),
    },

    "save_models": bool(
        SAVE_MODELS
    ),

    "save_preprocessors": bool(
        SAVE_PREPROCESSORS
    ),

    "save_oof_probabilities": bool(
        SAVE_OOF_PROBABILITIES
    ),
}

with open(
    OUTPUT_DIR
    / "12_run_configuration.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        run_configuration,
        file,
        ensure_ascii=False,
        indent=2,
    )

# ------------------------------------------------------------

# ------------------------------------------------------------

saved_files = sorted(
    OUTPUT_DIR.glob("*")
)

print("=" * 90)
print("All outputs saved successfully in:")
print(OUTPUT_DIR)

print("\nSaved files:")

for saved_file in saved_files:
    print(
        " -",
        saved_file.name,
    )

print("=" * 90)

All outputs saved successfully in:
<historical-output-root>/02B_legacy_RF_grouped_well_CV/full

Saved files:
 - 00_class_mapping.csv
 - 01_fold_assignment_summary.csv
 - 02_fold_metrics.csv
 - 03_grouped_metric_summary.csv
 - 04_per_class_metrics.csv
 - 05_per_well_metrics.csv
 - 06_feature_importance_by_fold.csv
 - 07_mean_feature_importance.csv
 - 08_resampling_audit_all_folds.csv
 - 09_oof_summary.csv
 - 10_official_test_vs_grouped_oof.csv
 - 11_run_manifest.csv
 - 12_run_configuration.json
 - artifacts
 - feature_importance
 - fold_metrics
 - oof_predictions
 - plots
 - resampling_audits
